## Launching colab kernel

Single chip
```
colab launch  //third_party/py/torchtitan:torchtitan_colab \
 --xm_resource_alloc=cloud-dynamic/cmcs-xm \
 --accelerator=vlp:1x1  \
 --label="$USER - vlp1x1 `date '+%Y-%m-%d %H:%M:%S'`"
```


In [ ]:
%%writefile /tmp/debug_model.toml
[job]
dump_folder = "./outputs"
description = "Qwen 3 debug model training"

[profiling]
enable_profiling = false
save_traces_folder = "profile_trace"
profile_freq = 100

[metrics]
log_freq = 1
enable_tensorboard = false
save_tb_folder = "tb"

[model]
name = "qwen3"
flavor = "debugmodel"
hf_assets_path = "./assets/hf/Qwen3-0.6B"
# converters = ["float8"]

[optimizer]
name = "AdamW"
lr = 3e-4
eps = 1e-8

[lr_scheduler]
warmup_steps = 2  # lr scheduler warm up, 20% total steps

[training]
local_batch_size = 4
seq_len = 128
max_norm = 1.0  # grad norm clipping
steps = 10
dataset = "c4_test"  # supported datasets: c4_test (2K), c4 (177M)

[parallelism]
data_parallel_replicate_degree = 1
data_parallel_shard_degree = -1
fsdp_reshard_after_forward = "default" # default / never / always
tensor_parallel_degree = 1
context_parallel_degree = 1

[checkpoint]
enable = false
folder = "checkpoint"
interval = 500
last_save_model_only = false
export_dtype = "float16"
async_mode = "disabled" # ["disabled", "async", "async_with_pinned_mem"]

[activation_checkpoint]
mode = "selective"  # ["none", "selective", "full"]
selective_ac_option = "op"  # "int" = ac every positive int layer or 'op', ac based on ops policy

[compile]
enable=false
components = ["model", "loss"]

[quantize.linear.float8]
enable_fsdp_float8_all_gather = false
precompute_float8_dynamic_scale_for_fsdp = false
filter_fqns = ["output"]


Writing /tmp/debug_model.toml


In [ ]:
import torchtitan.experiments.tpu.train_minimal

import torchtitan.config

num_devices = 1 # this is used for tp degree

config_manager = torchtitan.config.ConfigManager()
config = config_manager.parse_args([
    "--job.config_file=/tmp/debug_model.toml",
    "--model.name=qwen3_tpu",
    "--model.hf_assets_path=/cns/is-d/home/torch-tpu-xm/tests/assets/tokenizer",
    "--training.dataset_path=/cns/is-d/home/torch-tpu-xm/tests/assets/c4_test",
    "--training.seq_len=128",
    "--training.dataset=c4_test",
    "--parallelism.data_parallel_shard_degree=1",
    f"--parallelism.tensor_parallel_degree={num_devices}"
])
config.training.steps = 20

for k, v in config.__dict__.items():
  print(f"{k} -> {v}\n")

job -> Job(config_file='/tmp/debug_model.toml', dump_folder='./outputs', description='Qwen 3 debug model training', print_args=False)

profiling -> Profiling(enable_profiling=False, save_traces_folder='profile_trace', profile_freq=100, enable_memory_snapshot=False, save_memory_snapshot_folder='memory_snapshot')

metrics -> Metrics(log_freq=1, enable_tensorboard=False, disable_color_printing=False, save_tb_folder='tb', save_for_all_ranks=False, enable_wandb=False)

model -> Model(name='qwen3_tpu', flavor='debugmodel', hf_assets_path='/cns/is-d/home/torch-tpu-xm/tests/assets/tokenizer', tokenizer_path=None, converters=[], print_after_conversion=False)

optimizer -> Optimizer(name='AdamW', lr=0.0003, beta1=0.9, beta2=0.95, eps=1e-08, weight_decay=0.1, implementation='fused', early_step_in_backward=False)

lr_scheduler -> LRScheduler(warmup_steps=2, decay_ratio=None, decay_type='linear', min_lr_factor=0.0)

training -> Training(dataset='c4_test', dataset_path='/cns/is-d/home/torch-tpu-xm/t

In [ ]:
import torch
from torch_tpu._internal import profiler
import os

output_dir = os.path.join("/tmp", "xprof")
os.makedirs(output_dir, exist_ok=True)

with profiler.profile(
    activities=[
        profiler.ProfilerActivity.CPU,
        profiler.ProfilerActivity.TPU,
        ],
    on_trace_ready=profiler.xprof_trace_handler(dir_name=output_dir),
  ):
  torchtitan.experiments.tpu.train_minimal.start_trainer(
        torch.device("tpu"), 0, 1, config)
  print("done")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: tpu
Registered Python module for 'tpu'.


Device type: tpu, Device index: default
[titan] 2025-11-10 10:43:29,114 - root - INFO - parallel_dims: ParallelDims(dp_replicate=1, dp_shard=1, cp=1, tp=1, pp=1, ep=1, etp=1, world_size=1, _world_mesh=None)
[titan] 2025-11-10 10:43:29,925 - root - INFO - Loading tokenizer from tokenizer.json
[titan] 2025-11-10 10:43:33,119 - root - INFO - Preparing c4_test dataset from /cns/is-d/home/torch-tpu-xm/tests/assets/c4_test
[titan] 2025-11-10 10:43:38,094 - dill - INFO - D2: <dict object at 0x31ebed9c90c0>
[titan] 2025-11-10 10:43:38,095 - dill - INFO - T4: <class 'datasets.packaged_modules.json.json.JsonConfig'>
[titan] 2025-11-10 10:43:38,096 - dill - INFO - # T4
[titan] 2025-11-10 10:43:38,097 - dill - INFO - D2: <dict object at 0x31ebf7a23500>
[titan] 2025-11-10 10:43:38,098 - dill - INFO - T4: <class 'datasets.data_files.DataFilesDict'>
[titan] 2025-11-10 10:43:38,099 - dill - INFO - # T4
[titan] 2025-11-10 10:43:38,099 - dill - INFO - T4: <class 'datasets.splits.NamedSplit'>
[titan] 202

Generating train split: 0 examples [00:00, ? examples/s]

[titan] 2025-11-10 10:43:42,834 - dill - INFO - Si: range(0, 2000)
[titan] 2025-11-10 10:43:42,835 - dill - INFO - F2: <function _eval_repr at 0x31ebecc794e0>
[titan] 2025-11-10 10:43:42,836 - dill - INFO - # F2
[titan] 2025-11-10 10:43:42,837 - dill - INFO - # Si
[titan] 2025-11-10 10:43:42,840 - root - INFO - Building qwen3_tpu debugmodel with Qwen3ModelArgs(_enforced='This field is used to enforce all fields have defaults.', dim=256, n_layers=3, n_heads=16, n_kv_heads=8, vocab_size=2008, head_dim=128, hidden_dim=256, norm_eps=1e-06, rope_theta=1000000, qk_norm=True, max_seq_len=128, depth_init=True, use_flex_attn=False, attn_mask_type='causal', eos_id=151645, enable_weight_tying=True, moe_enabled=False, moe_inter_dim=768, moe_args=MoEArgs(num_experts=8, num_shared_experts=1, score_func='sigmoid', route_norm=False, route_scale=1.0, score_before_experts=True, top_k=1, use_grouped_mm=True, load_balance_coeff=0.001))
[titan] 2025-11-10 10:43:42,927 - root - INFO - Total parameter count:

/export/hda3/borglet/remote_hdd_fs_dirs/0.jialeic_xids-205535062-1-jia_205535062.1.kernel.jialeic.5253060022164.14b334fb3717c109/mount/server/torchtitan_colab.runfiles/google3/torchtitan/experiments/tpu/qwen3/model/model.py:102: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at blaze-out/haswell-opt/bin/third_party/py/torch/aten/src/ATen/core/TensorBody.h:494.)
  .reshape(bs, slen, n_kv_heads * n_rep, head_dim)
/export/hda3/borglet/remote_hdd_fs_dirs/0.jialeic_xids-205535062-1-jia_205535062.1.kernel.jialeic.5253060022164.14b334fb3717c109/mount/server/torchtitan_colab.runfiles/google3/torc

[titan] 2025-11-10 10:43:46,257 - root - INFO - Step 1/20 | Loss: 8.0662 | Throughput: 4379.24 tokens/sec


/export/hda3/borglet/remote_hdd_fs_dirs/0.jialeic_xids-205535062-1-jia_205535062.1.kernel.jialeic.5253060022164.14b334fb3717c109/mount/server/torchtitan_colab.runfiles/google3/torchtitan/experiments/tpu/train_minimal.py:263: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at blaze-out/haswell-opt/bin/third_party/py/torch/aten/src/ATen/core/TensorBody.h:494.)
  loss.item(),


[titan] 2025-11-10 10:43:47,092 - root - INFO - Step 2/20 | Loss: 7.9575 | Throughput: 38367.79 tokens/sec
[titan] 2025-11-10 10:43:47,419 - root - INFO - Step 3/20 | Loss: 7.6229 | Throughput: 42534.54 tokens/sec
[titan] 2025-11-10 10:43:47,446 - root - INFO - Step 4/20 | Loss: 7.5056 | Throughput: 40552.23 tokens/sec
[titan] 2025-11-10 10:43:47,472 - root - INFO - Step 5/20 | Loss: 7.3848 | Throughput: 42833.17 tokens/sec
[titan] 2025-11-10 10:43:47,498 - root - INFO - Step 6/20 | Loss: 7.0638 | Throughput: 43168.10 tokens/sec
[titan] 2025-11-10 10:43:47,523 - root - INFO - Step 7/20 | Loss: 6.8220 | Throughput: 48203.90 tokens/sec
[titan] 2025-11-10 10:43:47,548 - root - INFO - Step 8/20 | Loss: 7.3044 | Throughput: 44660.16 tokens/sec
[titan] 2025-11-10 10:43:47,574 - root - INFO - Step 9/20 | Loss: 6.3431 | Throughput: 44228.77 tokens/sec
[titan] 2025-11-10 10:43:47,600 - root - INFO - Step 10/20 | Loss: 6.2299 | Throughput: 44244.26 tokens/sec
[titan] 2025-11-10 10:43:47,626 - ro

In [ ]:
from torchtitan.experiments.tpu.profiling import upload_xprof_xplane_pb_files_from_dir

upload_xprof_xplane_pb_files_from_dir(output_dir)

[titan] 2025-11-10 10:43:51,556 - absl - INFO - Start execution cell: 8sPW7-hLc-91, notebook: /piper/depot/google3/torchtitan/experiments/tpu/colab/torchtpu_profile_1_tpu.ipynb?workspaceId=jialeic:CS-__init__-2025-11-07_122017::citc
Searching for .xplane.pb files in: /tmp/xprof
Found 1 .xplane.pb files:
Pick the latest one: /tmp/xprof/plugins/profile/2025_11_10_10_43_48/b0574543a6b57fa1-4c712c59203.borgtask.google.com.xplane.pb
Successfully loaded and parsed: /tmp/xprof/plugins/profile/2025_11_10_10_43_48/b0574543a6b57fa1-4c712c59203.borgtask.google.com.xplane.pb
Added hostname to response: tmp_xprof_plugins_profile_2025_11_10_10_43_48_b0574543a6b57fa1-4c712c59203.borgtask.google.com.xplane.pb
Uploading 1 profiles to Xprof...
[titan] 2025-11-10 10:43:53,670 - absl - INFO - Xprof session ID: http://xprof/trace_viewer.html?session_id=jialeic-3156895698647582573
Successfully uploaded profiles to Xprof!
View at: http://xprof/?session_id=jialeic-3156895698647582573


'jialeic-3156895698647582573'

[titan] 2025-11-10 10:43:53,759 - absl - INFO - End execution cell: 8sPW7-hLc-91, notebook: /piper/depot/google3/torchtitan/experiments/tpu/colab/torchtpu_profile_1_tpu.ipynb?workspaceId=jialeic:CS-__init__-2025-11-07_122017::citc, elapsed: 2.203(s)
